# Enemy power budget model

The notebook keeps balance assumptions and experiments visible. Reusable implementation details live in `enemy_power_budget_lib.py`.


## 1. Setup

This cell locates the project and imports the balance library. The library uses only the Python standard library and Jupyter's built-in display API.


In [1]:
from dataclasses import replace
from pathlib import Path
import importlib
import sys

for candidate in (Path.cwd(), *Path.cwd().parents):
    tools_path = candidate / "misc/tools"
    if (tools_path / "enemy_power_budget_lib.py").exists():
        PROJECT_ROOT = candidate
        if str(tools_path) not in sys.path:
            sys.path.insert(0, str(tools_path))
        break
else:
    raise FileNotFoundError("Could not find misc/tools/enemy_power_budget_lib.py")

import enemy_power_budget_lib as budget_lib
importlib.reload(budget_lib)  # Pick up library edits without restarting the kernel.

from enemy_power_budget_lib import (
    BalanceRules,
    REPORT_COLUMNS,
    build_tier_report,
    load_enemies,
    load_player_armor_references,
    show_table,
    synchronize_rules,
    trait_category,
    trait_power_breakdown,
)

ENEMY_DEFINITIONS_PATH = PROJECT_ROOT / "misc/entities/Enemies.csv"
ARMOR_DEFINITIONS_PATH = PROJECT_ROOT / "misc/entities/Armor.csv"


## 2. Domain configuration

`BalanceRules` is an immutable configuration object. All values intended for tuning are collected in `RULES`; `dataclasses.replace` can create experimental copies without changing these defaults.

`enemy_ttk_range` describes how many player hits an enemy should survive. `player_ttk_range` describes how many enemy hits the reference player should survive.


In [2]:
# RULES_START: managed by the synchronization cell at the end of this notebook.
RULES = BalanceRules(
    base_tier_power=100,
    power_gained_per_tier=50,
    power_unit=10,
    hp_per_power_unit=2,
    damage_per_power_unit=1,
    damage_range_fraction=0.20,
    characteristic_weights={
        "hp": 0.30,
        "damage": 0.20,
        "range": 0.15,
        "attack_speed": 0.08,
        "dodge": 0.07,
        "movement_speed": 0.06,
        "armor": 0.05,
        "accuracy": 0.02,
        "elemental_effect": 0.05,
        "resistance_or_vulnerability": 0.02,
    },
    enemy_ttk_range=(2, 4),
    armored_enemy_ttk_range=(3, 5),
    player_hp_base=20,
    player_hp_per_level=6,
    player_health_coefficient=0.15,
    player_health_reference=0,
    player_ttk_range=(12, 30),
)
# RULES_END

TIERS_TO_ANALYZE = (1, 3, 5, 8)
ZERO_COST_TRAITS = {"STATIC", "NEU"}


## 3. Source data

Enemy traits and the rightmost `csv_*` report columns come from `Enemies.csv`. The player armor reference is the mean midpoint of armor ranges at the corresponding `Peak` in `Armor.csv`. Reference player level equals enemy tier; HP is calculated from that level and the progression parameters in `RULES`.


In [3]:
enemies = load_enemies(ENEMY_DEFINITIONS_PATH)
player_armor_by_tier = load_player_armor_references(ARMOR_DEFINITIONS_PATH)

def report_for(tier, model=RULES):
    return build_tier_report(tier, enemies, player_armor_by_tier, model)

print(f"Loaded {len(enemies)} enemies and armor references for {len(player_armor_by_tier)} tiers.")


Loaded 20 enemies and armor references for 8 tiers.


## 4. Trait pricing

Trait prices are percentages of the tier's available power. The category mapping and arithmetic live in the library; inspect a concrete enemy in section 6 when tuning a trait.


## 5. Tier reports and the two TTK directions

- `enemy_ttk_hits`: player hits required to remove the calculated `target_hp`, after the enemy's `csv_armor`.
- `player_ttk_hits`: attacks from this calculated enemy required to remove `player_hp_reference`, after same-tier `player_armor_reference`.

Both are measured in successful hits rather than seconds. Dodge, accuracy, attack speed, effects, and damage randomness are intentionally excluded. `player_damage_reference` is the mean calculated enemy damage in the tier, so no CSV enemy damage participates in either calculated TTK.


In [4]:
tier_reports = {tier: report_for(tier) for tier in TIERS_TO_ANALYZE}

for tier, report in tier_reports.items():
    print(f"Tier {tier}")
    show_table(report, REPORT_COLUMNS)


Tier 1


enemy,enemy_ttk_hits,enemy_ttk_target,enemy_ttk_status,player_ttk_hits,player_ttk_target,player_ttk_status,traits,available_power,trait_power,remaining_power,target_hp,target_average_damage,player_damage_reference,player_level_reference,player_hp_reference,player_armor_reference,csv_hp,csv_average_damage,csv_armor,csv_total_power
Cave Snake,2.99,2–4,within target,17.11,12–30,within target,"POISON, DEX+",100,12.00,88.00,10.56,3.52,3.53,1,26.00,2.00,8.00,6.00,0.00,112.00
Cave Rat,3.33,2–4,within target,13.54,12–30,within target,F-,100,2.00,98.00,11.76,3.92,3.53,1,26.00,2.00,14.00,7.00,0.00,142.00
Bat,2.68,2–4,within target,22.41,12–30,within target,"SPD+, ATK+, DEX+",100,21.00,79.00,9.48,3.16,3.53,1,26.00,2.00,8.00,5.00,0.00,111.00


Tier 3


enemy,enemy_ttk_hits,enemy_ttk_target,enemy_ttk_status,player_ttk_hits,player_ttk_target,player_ttk_status,traits,available_power,trait_power,remaining_power,target_hp,target_average_damage,player_damage_reference,player_level_reference,player_hp_reference,player_armor_reference,csv_hp,csv_average_damage,csv_armor,csv_total_power
Carrion Feeder,3.10,2–4,within target,12.67,12–30,within target,—,200,0,200,24.00,8.00,7.73,3,38.00,5.00,77.00,13.00,0.00,515.00
Venom Centipede,2.95,2–4,within target,14.62,12–30,within target,POISON,200,10.00,190.00,22.80,7.60,7.73,3,38.00,5.00,77.00,13.00,0.00,525.00
Acid Centipede,2.95,2–4,within target,14.62,12–30,within target,ACID,200,10.00,190.00,22.80,7.60,7.73,3,38.00,5.00,77.00,13.00,0.00,525.00


Tier 5


enemy,enemy_ttk_hits,enemy_ttk_target,enemy_ttk_status,player_ttk_hits,player_ttk_target,player_ttk_status,traits,available_power,trait_power,remaining_power,target_hp,target_average_damage,player_damage_reference,player_level_reference,player_hp_reference,player_armor_reference,csv_hp,csv_average_damage,csv_armor,csv_total_power
Cave Spider,3.54,3–5,within target,n/a,12–30,damage blocked by armor,"ACID, RNG(4), PER+, DEX-, SPD-, ATK-",300,129.00,171.00,20.52,6.84,8.80,5,50.00,10.00,85.00,18.50,3.00,739.00
Repair Drone,6.37,3–5,above target,250.00,12–30,above target,"ARM, SPD+, F+, P+",300,45.00,255.00,30.60,10.20,8.80,5,50.00,10.00,85.00,16.00,4.00,630.00
Mechatron,n/a,3–5,damage blocked by armor,n/a,12–30,damage blocked by armor,"ARM, F+, P+, SPD-, DEX-",300,66.00,234.00,28.08,9.36,8.80,5,50.00,10.00,129.00,19.00,9.00,901.00


Tier 8


enemy,enemy_ttk_hits,enemy_ttk_target,enemy_ttk_status,player_ttk_hits,player_ttk_target,player_ttk_status,traits,available_power,trait_power,remaining_power,target_hp,target_average_damage,player_damage_reference,player_level_reference,player_hp_reference,player_armor_reference,csv_hp,csv_average_damage,csv_armor,csv_total_power
Terminator,7.67,3–5,above target,n/a,12–30,damage blocked by armor,"ARM, DEX+, SPD+, F+, P+, FIRE",450,121.50,328.50,39.42,13.14,13.14,8,68.00,18.00,232.00,35.00,8.00,1631.50


## 6. Inspect traits and validate

Validation rejects overspent power and unknown traits. A blocked TTK is reported separately: in the simplified model, damage that does not exceed armor cannot kill the target.


In [5]:
ENEMY_TO_INSPECT = "Cave Spider"
selected = next((enemy for enemy in enemies if enemy.name == ENEMY_TO_INSPECT), None)
if selected is None:
    raise ValueError(f"Unknown enemy: {ENEMY_TO_INSPECT}")

show_table(
    trait_power_breakdown(selected.traits, selected.tier, RULES),
    ("trait", "category", "weight_percent", "power_cost"),
)

combined_report = [row for tier in TIERS_TO_ANALYZE for row in tier_reports[tier]]
all_traits = sorted({trait for enemy in enemies for trait in enemy.traits})
unpriced = [trait for trait in all_traits if trait_category(trait) is None]
unexpected = set(unpriced) - ZERO_COST_TRAITS

assert all(row["remaining_power"] >= 0 for row in combined_report)
assert not unexpected, f"Unknown traits need pricing: {sorted(unexpected)}"

for column, label in (
    ("enemy_ttk_hits", "Player damage is fully blocked by enemy armor"),
    ("player_ttk_hits", "Enemy damage is fully blocked by player armor"),
):
    blocked = [row["enemy"] for row in combined_report if row[column] is None]
    if blocked:
        print(f"{label}: {', '.join(blocked)}")

print("Intentionally zero-cost traits found:", ", ".join(unpriced))
print("Validation passed.")


trait,category,weight_percent,power_cost
ACID,elemental_effect,5.00,15.00
RNG(4),range,15.00,45.00
PER+,accuracy,2.00,6.00
DEX-,dodge,7.00,21.00
SPD-,movement_speed,6.00,18.00
ATK-,attack_speed,8.00,24.00


Player damage is fully blocked by enemy armor: Mechatron
Enemy damage is fully blocked by player armor: Cave Spider, Mechatron, Terminator
Intentionally zero-cost traits found: STATIC
Validation passed.


## 7. Experiment without mutation

`dataclasses.replace` creates a modified copy of `RULES`; the default configuration remains unchanged. Pass the copy to `report_for(..., model=experimental_rules)` to calculate an alternative scenario.

The model distinguishes calculated `target_*` values from source `csv_*` values. Changing a rule updates targets, both calculated TTK directions, and `csv_total_power`; it never rewrites the source CSV files.

### Conversion controls

| Control | Increasing it does |
| --- | --- |
| `hp_per_power_unit` | Produces more target HP, increasing enemy survivability TTK. Existing CSV HP also becomes cheaper in `csv_total_power`. |
| `damage_per_power_unit` | Produces more target damage, reducing player survivability TTK. Existing CSV damage becomes cheaper in `csv_total_power`. |
| `damage_range_fraction` | Widens the integer damage range around its rounded average without changing that average. |
| `power_unit` | Makes target HP and damage more expensive, while increasing the estimated power cost of CSV stats. |

### Tier progression controls

| Control | Increasing it does |
| --- | --- |
| `base_tier_power` | Adds power to every tier, including tier 1. |
| `power_gained_per_tier` | Makes progression steeper; it has no effect on tier 1. |

### Characteristic weights

Copy the dictionary before changing a weight. Increasing `hp` or `damage` changes their relative split. Increasing a trait category makes matching traits more expensive and leaves less power for both base stats.

### TTK controls

| Control | Meaning |
| --- | --- |
| `enemy_ttk_range` | Target player-hit range for enemies without armor. |
| `armored_enemy_ttk_range` | Target player-hit range for armored enemies. |
| `player_hp_base` | Level-independent part of reference player HP. |
| `player_hp_per_level` | HP gained per reference player level. |
| `player_health_coefficient` | Contribution of one point of the health characteristic. |
| `player_health_reference` | Health characteristic used for the reference build. |
| `player_ttk_range` | Target enemy-hit range for player survivability. |

The ranges change only status labels. Reference player level equals tier, and `player_hp_reference` is calculated from the HP controls above. Same-tier player armor references come from `Armor.csv`.


In [8]:
# Active experiment: reduce damage produced by each power unit.
experimental_rules = replace(RULES, damage_per_power_unit=1)
show_table(report_for(1, model=experimental_rules), REPORT_COLUMNS)

# Example: make ranged traits more expensive without mutating RULES.
# experimental_weights = RULES.characteristic_weights | {"range": 0.20}
# experimental_rules = replace(RULES, characteristic_weights=experimental_weights)
# show_table(report_for(5, model=experimental_rules), REPORT_COLUMNS)


enemy,enemy_ttk_hits,enemy_ttk_target,enemy_ttk_status,player_ttk_hits,player_ttk_target,player_ttk_status,traits,available_power,trait_power,remaining_power,target_hp,target_average_damage,player_damage_reference,player_level_reference,player_hp_reference,player_armor_reference,csv_hp,csv_average_damage,csv_armor,csv_total_power
Cave Snake,2.99,2–4,within target,17.11,12–30,within target,"POISON, DEX+",100,12.00,88.00,10.56,3.52,3.53,1,26.00,2.00,8.00,6.00,0.00,112.00
Cave Rat,3.33,2–4,within target,13.54,12–30,within target,F-,100,2.00,98.00,11.76,3.92,3.53,1,26.00,2.00,14.00,7.00,0.00,142.00
Bat,2.68,2–4,within target,22.41,12–30,within target,"SPD+, ATK+, DEX+",100,21.00,79.00,9.48,3.16,3.53,1,26.00,2.00,8.00,5.00,0.00,111.00


## 8. Persist the selected experiment

This optional step promotes `experimental_rules` to the project defaults. It updates the managed `RULES` block in this notebook, the generated enemy-power block in `misc/game_balance.md`, and integer damage, HP, and armor values in `misc/entities/Enemies.csv`.

The final cell is disabled by default. After writing, reload the notebook from disk immediately and inspect `git diff` before committing.


In [7]:
# Safety switch: leave False while experimenting.
WRITE_CHANGES = False

if WRITE_CHANGES:
    synchronize_rules(
        experimental_rules,
        PROJECT_ROOT / "misc/tools/enemy_power_budget_python.ipynb",
        (PROJECT_ROOT / "misc/game_balance.md",),
        ENEMY_DEFINITIONS_PATH,
        ARMOR_DEFINITIONS_PATH,
    )
else:
    print("Dry run: no files changed.")
    print("Selected experiment:", experimental_rules)
    print("Set WRITE_CHANGES = True only when these values are ready to commit.")


Dry run: no files changed.
Selected experiment: BalanceRules(base_tier_power=100, power_gained_per_tier=50, power_unit=10, hp_per_power_unit=2, damage_per_power_unit=0.8, characteristic_weights={'hp': 0.3, 'damage': 0.2, 'range': 0.15, 'attack_speed': 0.08, 'dodge': 0.07, 'movement_speed': 0.06, 'armor': 0.05, 'accuracy': 0.02, 'elemental_effect': 0.05, 'resistance_or_vulnerability': 0.02}, enemy_ttk_range=(2, 4), armored_enemy_ttk_range=(3, 5), player_hp_base=20, player_hp_per_level=6, player_health_coefficient=0.15, player_health_reference=0, player_ttk_range=(12, 30))
Set WRITE_CHANGES = True only when these values are ready to commit.
